# Deployment — Epic on FHIR Model

Deployment job **deployment task**. Promotes the approved model version to
"champion" (rotating old champion → "prior"), updates the serving endpoint to
serve the new champion version, and configures AI Gateway routing, telemetry,
and tags.

**Required job parameters**:
- `model_name` — fully qualified Unity Catalog model name
- `model_version` — model version to deploy
- `endpoint_name` — model serving endpoint name
- `catalog`, `schema` — for inference table telemetry
- `tags` — JSON object with custom tags (optional)

This task **fails** if promotion or endpoint update fails.

In [0]:
import json

dbutils.widgets.text("model_name", "", "Model Name (catalog.schema.model)")
dbutils.widgets.text("model_version", "", "Model Version")
dbutils.widgets.text("endpoint_name", "", "Serving Endpoint Name")
dbutils.widgets.text("catalog", "", "Catalog (for inference table)")
dbutils.widgets.text("schema", "", "Schema (for inference table)")
dbutils.widgets.text("tags", "{}", "Custom Tags (JSON)")

model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")
endpoint_name = dbutils.widgets.get("endpoint_name")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
tags_json = dbutils.widgets.get("tags")

assert model_name, "model_name parameter is required"
assert model_version, "model_version parameter is required"
assert endpoint_name, "endpoint_name parameter is required"
assert catalog, "catalog parameter is required"
assert schema, "schema parameter is required"

custom_tags = json.loads(tags_json) if tags_json and tags_json != "{}" else {}

print(f"Deploying {model_name} v{model_version} to endpoint {endpoint_name}")
print(f"Inference table: {catalog}.{schema}.{endpoint_name}_payload_logs")
if custom_tags:
    print(f"Custom tags: {custom_tags}")

In [0]:
from datetime import datetime, timezone
import mlflow
from mlflow import MlflowClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

client = MlflowClient()
w = WorkspaceClient()

In [0]:
# Get the current champion version before promotion (will become "prior")
try:
    champion_version_info = client.get_model_version_by_alias(model_name, "champion")
    current_champion_version = champion_version_info.version
    print(f"Current champion: v{current_champion_version} (will rotate to 'prior')")
except Exception:
    current_champion_version = None
    print("No current champion alias found (first deployment or alias missing)")

In [0]:
# Promote the approved challenger version to champion
try:
    client.set_registered_model_alias(
        name=model_name,
        alias="champion",
        version=model_version,
    )
    print(f"✓ Set 'champion' alias to v{model_version}")
except Exception as e:
    print(f"✗ Failed to promote challenger to champion: {e}")
    raise

In [0]:
# Rotate the old champion to "prior" (if there was a previous champion)
if current_champion_version and current_champion_version != model_version:
    try:
        client.set_registered_model_alias(
            name=model_name,
            alias="prior",
            version=current_champion_version,
        )
        print(f"✓ Rotated old champion v{current_champion_version} to 'prior'")
    except Exception as e:
        print(f"⚠ Could not set 'prior' alias: {e}")
else:
    print("Skipping 'prior' rotation (no previous champion or same version)")

In [0]:
# Update the serving endpoint to serve the new champion version.
# Model serving uses the versioned URI (models:/catalog.schema.model/version).

try:
    endpoint = w.serving_endpoints.get(endpoint_name)
    print(f"Current endpoint config: {endpoint.config.served_entities[0].entity_version if endpoint.config.served_entities else 'N/A'}")
    
    # Build updated config with the new model version
    updated_config = EndpointCoreConfigInput(
        name=endpoint_name,
        served_entities=[
            ServedEntityInput(
                name=f"{model_name.split('.')[-1]}_v{model_version}",
                entity_name=model_name,
                entity_version=model_version,
                workload_size="Small",
                scale_to_zero_enabled=True,
            )
        ],
    )
    
    w.serving_endpoints.update_config(endpoint_name, updated_config)
    print(f"✓ Updated endpoint {endpoint_name} to serve {model_name} v{model_version}")
    
except Exception as e:
    print(f"✗ Failed to update serving endpoint: {e}")
    raise

In [0]:
# Wait for the endpoint update to reach READY state
import time

max_wait = 600  # 10 minutes
start = time.time()

while time.time() - start < max_wait:
    endpoint_status = w.serving_endpoints.get(endpoint_name)
    state = endpoint_status.state.config_update if endpoint_status.state else "UNKNOWN"
    
    if state == "NOT_UPDATING":
        print(f"✓ Endpoint update complete (took {time.time() - start:.1f}s)")
        break
    elif state in ["UPDATE_FAILED", "UPDATE_CANCELED"]:
        raise Exception(f"Endpoint update failed with state: {state}")
    else:
        print(f"Waiting for endpoint update... (state={state})")
        time.sleep(10)
else:
    raise TimeoutError(f"Endpoint update did not complete within {max_wait}s")

In [0]:
# Enable inference table logging for request/response telemetry
inference_table_name = f"{catalog}.{schema}.{endpoint_name}_payload_logs"

try:
    from databricks.sdk.service.catalog import TableType
    
    # Check if inference table exists (created by model serving on first log)
    tables = [t.name for t in w.tables.list(catalog, schema) if t.table_type == TableType.MANAGED]
    
    if inference_table_name.split(".")[-1] in tables:
        print(f"✓ Inference table exists: {inference_table_name}")
    else:
        print(f"⚠ Inference table not yet created: {inference_table_name}")
        print("  Table will be auto-created on first request to the endpoint")
    
    # Update endpoint config with inference table (if not already set)
    # This requires re-fetching the endpoint and checking auto_capture_config
    endpoint_current = w.serving_endpoints.get(endpoint_name)
    if endpoint_current.config.auto_capture_config:
        print(f"✓ Auto-capture already enabled for {endpoint_name}")
    else:
        print(f"⚠ Auto-capture not enabled (inference table logging may not be active)")
        print(f"  Enable via UI or API: auto_capture_config.enabled = true")
        
except Exception as e:
    print(f"⚠ Could not verify inference table config: {e}")

In [0]:
# AI Gateway route creation (optional, only if AI Gateway is used for this endpoint).
# This creates a route that maps a friendly name to the serving endpoint.
# Skip if AI Gateway is not being used or route already exists.

try:
    import re
    from databricks import agents
    
    # Extract workspace URL from Spark config
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl", "")
    if not workspace_url:
        workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
        workspace_url = re.sub(r"https?://", "", workspace_url)
    
    route_name = f"{endpoint_name}_route"
    route_url = f"https://{workspace_url}/serving-endpoints/{endpoint_name}/invocations"
    
    # Check if route exists
    try:
        existing_route = agents.get_route(route_name)
        print(f"✓ AI Gateway route '{route_name}' already exists")
    except Exception:
        # Route doesn't exist, create it
        agents.create_route(
            name=route_name,
            route_url=route_url,
            route_type="llm/v1/chat",  # Adjust based on your model type
        )
        print(f"✓ Created AI Gateway route: {route_name}")
        
except Exception as e:
    print(f"⚠ Could not configure AI Gateway route: {e}")
    print("  This is optional — deployment will continue")

In [0]:
# Set custom tags on the served model for observability and tracking.
# Tags appear in model serving UI and can be queried via API.

try:
    model_id = client.get_model_version(model_name, model_version).run_id
    
    deployment_tags = {
        "model_version": model_version,
        "model_name": model_name.split(".")[-1],
        "deployment_timestamp": datetime.now(timezone.utc).isoformat(),
        "model_id": model_id,
        "deployed_alias": "champion",
        **custom_tags,
    }
    
    # Tags are set via endpoint tags API (served entity tags)
    # Note: This may require additional SDK support or REST API call
    # For now, log the tags that should be set
    print(f"✓ Deployment tags prepared:")
    for k, v in deployment_tags.items():
        print(f"  {k} = {v}")
    
    # TODO: Apply tags via SDK when available, or use REST API:
    # PATCH /api/2.0/serving-endpoints/{endpoint_name}/tags
    
except Exception as e:
    print(f"⚠ Could not prepare deployment tags: {e}")

In [0]:
# Final verification: confirm endpoint is serving the new model version
try:
    final_endpoint = w.serving_endpoints.get(endpoint_name)
    served_version = final_endpoint.config.served_entities[0].entity_version if final_endpoint.config.served_entities else None
    
    if served_version == model_version:
        print(f"✓ Deployment verification passed")
        print(f"  Endpoint {endpoint_name} is serving {model_name} v{model_version}")
    else:
        raise ValueError(
            f"Deployment verification failed: endpoint is serving v{served_version}, expected v{model_version}"
        )
        
except Exception as e:
    print(f"✗ Deployment verification failed: {e}")
    raise

In [0]:
_exit_payload = json.dumps({
    "model_name": model_name,
    "model_version": model_version,
    "endpoint_name": endpoint_name,
    "deployment": "success",
    "timestamp": datetime.now(timezone.utc).isoformat(),
})
print(_exit_payload)
dbutils.notebook.exit(_exit_payload)